<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification</b></h1>
</div>

## Requirements and Approach

This document defines the experimental specification for a controlled comparison of one-hidden-layer MLP classifiers on MNIST.

The design fixes the dataset, preprocessing, optimization protocol, training duration, evaluation procedure, and reproducibility controls so that hidden-layer width remains the principal experimental variable. Requirements cover data integrity, model architecture, optimization stability, test-set evaluation, confidence analysis, output generation, and numerical validation.

## Global Requirements

| Requirement | Implemented value |
| --- | --- |
| Source data | Local MNIST IDX files |
| Input representation | Flattened $28\times28=784$ float vector |
| Pixel normalization | divide by 255 to $[0,1]$ |
| Hidden widths | 128, 256, 512 |
| Hidden block | Linear → BatchNorm1d → ReLU |
| Output | 10 logits |
| Training loss | Cross-Entropy |
| Optimizer | Adam |
| Epochs | 10 |
| Batch size | 512 |
| Learning rate | $10^{-3}$ |
| Weight decay | $10^{-3}$ |
| Evaluation | test accuracy + confidence-selective analysis |

## 1. Validate MNIST Data and Output Paths

**Approach:** validate the four expected IDX filenames under the repository-relative data directory and create the figures directory.

**Acceptance:** no required file is missing and output storage is writable.

## 2. Build the MNIST Dataset and Mini-Batch Pipeline

**Approach:** implement an IDX-backed Dataset, normalize each image by $255$, return integer labels, and use a custom collate function plus DataLoader.

**Acceptance:** batches contain float images and integer labels with consistent first dimension.

## 3. Load and Validate Training and Testing Data

**Approach:** instantiate training/testing datasets/loaders and inspect a sample.

**Acceptance:** sizes are 60,000/10,000; sample shape is $(784,)$; pixels lie in $[0,1]$; labels lie in $[0,9]$.

## 4. Visualize Representative MNIST Samples

**Approach:** reshape flattened samples to $28\times28$, display 12 examples, and save the figure.

**Acceptance:** the diagnostic image exists and labels match the samples shown.

## 5. Define the One-Hidden-Layer MLP Classifier

**Approach:** define a reusable PyTorch module with Linear(784,H), BatchNorm1d(H), ReLU, Linear(H,10). Return Cross-Entropy loss in training mode and Softmax-decoded class/confidence in evaluation mode.

**Acceptance:** one class logit is produced for each of 10 digits.

## 6. Verify the Model Architecture and Forward Pass

**Approach:** run one real training batch through a 256-hidden-unit model.

**Acceptance:** finite scalar loss and logit shape $(B,10)$.

## 7. Define Training and Evaluation Utilities

**Approach:** isolate batch conversion, train loop, and no-gradient evaluation in reusable functions.

**Acceptance:** training returns exactly one mean loss per epoch; evaluation returns predictions, confidence, labels, logits, and scalar accuracy.

## 8. Train the 128-, 256-, and 512-Neuron Models

**Approach:** reset the PyTorch seed for each width, train under identical optimizer settings, then evaluate on the same test loader.

**Acceptance:** models, 10-epoch histories, and evaluation results exist for 128, 256, and 512.

## 9. Compare Training Loss and Test Accuracy

**Approach:** plot all three loss histories on one figure and print final loss/test accuracy with submitted reference-loss values.

**Acceptance:** comparison figure exists and all reported live values are finite.

## 10. Visualize Predictions and Confidence Scores

**Approach:** run inference on representative test images and display image/true label/predicted label/confidence plus full Softmax bars.

**Acceptance:** one saved prediction figure exists for each hidden width.

## 11. Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy

**Approach:** threshold maximum Softmax confidence. Define correctness as the positive event, then compute $TP$, $FP$, and $FN$ correctly before deriving precision/recall. Accepted accuracy is $TP/N$.

**Acceptance:** curves contain finite values wherever denominators are defined, with all metrics bounded by $[0,1]$.

## 12. Analyze the Best Model and Save Evaluation Curves

**Approach:** identify the width with minimum final training loss to match the submitted-lab criterion and plot Precision–Recall plus Accepted-Accuracy–Recall.

**Acceptance:** the selected width, final loss, and test accuracy are reported and the curve figure exists.

## 13. Run Numerical and Output-file Validation Checks

**Approach:** run final structural, numerical, prediction, confidence, history-length, and filesystem checks.

**Acceptance:** any inconsistency raises an explicit error; successful execution prints the final validation message.

## Requirement-to-Code Traceability

| Task | Main implementation location |
| ---: | --- |
| 1 | data/output-path validation cell |
| 2 | `MNISTDataset` + `build_dataset_and_loader()` |
| 3 | dataset/loader instantiation and validation cell |
| 4 | MNIST sample visualization cell |
| 5 | `MNISTClassifier` |
| 6 | architecture/forward-pass verification cell |
| 7 | `prepare_batch()`, `train_model()`, `evaluate_model()` |
| 8 | three-model training/evaluation loop |
| 9 | training-loss comparison + performance table |
| 10 | `plot_prediction_examples()` |
| 11 | `compute_confidence_curves()` |
| 12 | best-analysis-model curve cell |
| 13 | final validation cell |

Every numbered task in the Implementation notebook has executable code immediately below its matching heading.